# Interactive Shape Blend Splines Demo

This notebook demonstrates the **non-rational** Shape Blend Spline (SBS) formulation used in the package:

$$\mathbf{C}(t)=\sum_{j=0}^{k-1} W_j(t)\,\mathbf{S}_j(t).$$

- $\mathbf{S}_j(t)$ are whole parametric shape functions evaluated at the same global parameter $t$
- $W_j(t)$ are partition-of-unity weights built from smooth **polynomial** step-function differences
- there is **no rational denominator**
- `alpha` controls the transition from global smoothing to strong local shape identity

The examples below use the package API directly and focus on the main SBS story: **open and closed curves assembled from whole geometric primitives**.


In [ ]:
# Install dependencies on Google Colab only
import subprocess
import sys

def colab_install():
    try:
        import google.colab  # noqa: F401
    except ImportError:
        return
    subprocess.check_call([sys.executable, "-m", "pip", "install", "numpy", "matplotlib", "ipywidgets", "--quiet"])
    subprocess.check_call([sys.executable, "-m", "pip", "install", "git+https://github.com/QL-UoHull/Shape-Blend-Splines.git", "--quiet"])

colab_install()


In [ ]:
from functools import partial

import matplotlib.pyplot as plt
import numpy as np

from shape_blend_splines import (
    ControlPointSpline,
    PeriodicShapeBlendSpline,
    ShapeBlendSpline,
    ShapeBlender,
)
from shape_blend_splines.shapes import (
    circle_arc,
    ellipse_arc,
    from_control_points,
    line_segment,
    rectangle_arc,
    star_arc,
    superellipse_arc,
)


## 1) Closed periodic SBS curve from whole primitives

Here the first and last shapes are treated as neighbours, so the weight functions wrap around the parameter domain and produce a closed curve.


In [ ]:
def closed_demo_shapes():
    shapes = [
        partial(circle_arc, radius=1.00),
        partial(ellipse_arc, a=1.45, b=0.75),
        partial(superellipse_arc, a=1.15, b=1.15, exponent=4.0),
        partial(rectangle_arc, width=2.0, height=1.6),
        partial(star_arc, outer_radius=1.25, inner_radius=0.50),
    ]
    labels = ["Circle", "Ellipse", "Superellipse", "Rectangle", "Star"]
    return shapes, labels

closed_shapes, closed_labels = closed_demo_shapes()
t_closed = np.linspace(0.0, 1.0, 900, endpoint=False)
closed_sbs = PeriodicShapeBlendSpline(closed_shapes, locality=3.0)
closed_pts = closed_sbs.evaluate(t_closed)


In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 6.0))
colors = plt.cm.tab10.colors
for j, (shape_fn, label) in enumerate(zip(closed_shapes, closed_labels)):
    sp = np.atleast_2d(shape_fn(t_closed))
    ax.plot(sp[:, 0], sp[:, 1], "--", lw=1.2, alpha=0.35, color=colors[j], label=label)
ax.plot(closed_pts[:, 0], closed_pts[:, 1], color="black", lw=2.8, label="Closed SBS")
ax.set_aspect("equal")
ax.set_title("Closed periodic SBS blend")
ax.legend(fontsize=8)
ax.set_xlabel("x")
ax.set_ylabel("y")
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
for ax, alpha in zip(axes, [0.8, 2.0, 7.0]):
    sbs = PeriodicShapeBlendSpline(closed_shapes, locality=alpha)
    pts = sbs.evaluate(t_closed)
    ax.plot(pts[:, 0], pts[:, 1], color="steelblue", lw=2.5)
    ax.set_aspect("equal")
    ax.set_title(rf"$\alpha={alpha}$")
    ax.grid(alpha=0.2)
fig.suptitle("Locality sweep for the same closed curve family", y=1.02)
plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 3.8), sharey=True)
for ax, alpha in zip(axes, [1.0, 3.0, 8.0]):
    W = PeriodicShapeBlendSpline(closed_shapes, locality=alpha).weights_at(t_closed)
    for j, label in enumerate(closed_labels):
        ax.plot(t_closed, W[j], lw=2, label=label)
    ax.set_title(rf"$\alpha={alpha}$")
    ax.set_xlabel("t")
    ax.set_ylim(-0.02, 1.02)
axes[0].set_ylabel(r"$W_j(t)$")
axes[0].legend(fontsize=7)
plt.tight_layout()
plt.show()


## 2) Open SBS sequence

SBS is not restricted to closed loops. The same package can blend an ordered sequence of open shapes, arcs, and freeform curves.


In [ ]:
freeform_pts = np.array([
    [-1.7, -0.3],
    [-1.0,  1.1],
    [ 0.3,  0.2],
    [ 1.7,  1.0],
])

open_shapes = [
    partial(line_segment, p0=(-1.8, -0.55), p1=(1.8, -0.45)),
    partial(circle_arc, center=(0.0, -0.15), radius=1.3,
            theta_start=0.95 * np.pi, theta_end=0.05 * np.pi),
    partial(ellipse_arc, center=(0.2, 0.15), a=1.55, b=0.85,
            theta_start=np.pi, theta_end=2.0 * np.pi),
    partial(from_control_points, control_pts=freeform_pts),
]
open_labels = ["Line", "Circle arc", "Ellipse arc", "Freeform"]
open_sbs = ShapeBlendSpline(open_shapes, locality=2.4)
t_open = np.linspace(0.0, 1.0, 700)
open_pts = open_sbs.evaluate(t_open)

fig, ax = plt.subplots(figsize=(7.5, 4.6))
colors = plt.cm.Set2.colors
for j, (shape_fn, label) in enumerate(zip(open_shapes, open_labels)):
    sp = np.atleast_2d(shape_fn(t_open))
    ax.plot(sp[:, 0], sp[:, 1], "--", lw=1.2, alpha=0.45, color=colors[j], label=label)
ax.plot(open_pts[:, 0], open_pts[:, 1], color="black", lw=2.7, label="Open SBS")
ax.set_aspect("equal")
ax.set_title("Open shape sequence")
ax.legend(fontsize=8)
ax.set_xlabel("x")
ax.set_ylabel("y")
plt.show()


## 3) Global weighted blend versus locality-aware SBS

A global blend is a useful baseline, but it behaves differently from SBS. Here both use the same component shapes; only the weighting philosophy changes.


In [ ]:
global_pts = ShapeBlender(closed_shapes, weights=[1, 1, 1, 1, 1]).evaluate(t_closed)
local_pts = PeriodicShapeBlendSpline(closed_shapes, locality=5.5).evaluate(t_closed)

fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.8))
for ax, pts, title in zip(axes, [global_pts, local_pts], ["Global weighted baseline", "Locality-aware closed SBS"]):
    ax.plot(pts[:, 0], pts[:, 1], color="black", lw=2.7)
    ax.set_aspect("equal")
    ax.set_title(title)
    ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()


## 4) Control-point workflow

The repository also keeps a familiar control-point interface for open and closed spline-style design tasks.


In [ ]:
control_pts = np.array([
    [0.0, 0.0],
    [1.0, 1.8],
    [3.0, 0.8],
    [4.0, 2.4],
    [5.5, 0.2],
])

open_cp = ControlPointSpline(control_pts, locality=2.0)
closed_cp = ControlPointSpline(control_pts, locality=2.0, closed=True)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.6))
axes[0].plot(*open_cp.evaluate(np.linspace(0.0, 1.0, 700)).T, color="steelblue", lw=2.5)
axes[0].plot(control_pts[:, 0], control_pts[:, 1], "o--", color="tomato", lw=1.2)
axes[0].set_title("Open control-point spline")
axes[0].set_aspect("equal")

closed_poly = np.vstack([control_pts, control_pts[:1]])
axes[1].plot(*closed_cp.evaluate(np.linspace(0.0, 1.0, 700, endpoint=False)).T, color="seagreen", lw=2.5)
axes[1].plot(closed_poly[:, 0], closed_poly[:, 1], "o--", color="tomato", lw=1.2)
axes[1].set_title("Closed control-point spline")
axes[1].set_aspect("equal")

for ax in axes:
    ax.set_xlabel("x")
    ax.set_ylabel("y")
plt.tight_layout()
plt.show()
